### Validation

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import statsmodels.api as sm


In [ ]:
# Define a function to run models and return performance metrics
def run_models(df):
    y = df["log_price"]
    X = df.drop(columns=["price", "log_price"])

    X = pd.get_dummies(X, drop_first=True)
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    results = {}

    # LASSO
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    lasso = LassoCV(cv=5, random_state=42)
    lasso.fit(X_train_s, y_train)
    y_pred_lasso = lasso.predict(X_test_s)

    results["LASSO"] = (
        np.sqrt(mean_squared_error(y_test, y_pred_lasso)),
        r2_score(y_test, y_pred_lasso)
    )

    # Random Forest
    rf = RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)

    results["Random Forest"] = (
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        r2_score(y_test, y_pred_rf)
    )

    # Gradient Boosting
    gbr = GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, random_state=42
    )
    gbr.fit(X_train, y_train)
    y_pred_gbr = gbr.predict(X_test)

    results["Gradient Boosting"] = (
        np.sqrt(mean_squared_error(y_test, y_pred_gbr)),
        r2_score(y_test, y_pred_gbr)
    )

    return pd.DataFrame(results, index=["RMSE", "R_squared"]).T


In [ ]:
# Load the cleaned Berlin 2025 Q1 dataset
results_berlin_time = run_models(df_berlin_time)
results_berlin_time


,RMSE,R_squared
LASSO,0.468572,0.418433
Random Forest,0.428148,0.514450
Gradient Boosting,0.422799,0.526507


#### Time validity – Berlin (earlier snapshot)

Model performance on the earlier Berlin snapshot remains close to the main sample results. While predictive accuracy decreases slightly, the relative ranking of models is unchanged, with tree-based models continuing to outperform LASSO. This suggests that the pricing relationships identified in the main analysis are stable over time.


In [14]:
# Load Geneva cleaned dataset
df_geneva = pd.read_csv("../data/processed/geneva_2025_clean.csv")

df_geneva.shape


results_geneva = run_models(df_geneva)
results_geneva


,RMSE,R_squared
LASSO,0.391019,0.529858
Random Forest,0.413918,0.473180
Gradient Boosting,0.397207,0.514858


Spatial validity – Geneva

When applied to Geneva, model performance differs from Berlin, reflecting structural differences between local Airbnb markets. Interestingly, the regularized LASSO model performs comparatively well, suggesting that simpler linear relationships capture pricing dynamics more effectively in this market. Nevertheless, Gradient Boosting remains competitive, indicating that nonlinear models still provide robust performance across cities. Overall, the results highlight that model performance and ranking may vary across locations, underlining the importance of local market characteristics.
